# 02 · Calibración del exponente pitagórico y de K y V del Elo

El pipeline arranca con K = 20 y V = 100 (sección 5.6). Acá se calibran con la liga: el exponente que mejor reproduce los puntos de cada tabla y el par (K, V) que minimiza el log-loss de la probabilidad pre-partido.

In [1]:
import sys, statistics, math
sys.path.insert(0, '.')
from datos import api, league_games, result, log_loss, elo_probabilities, m, RESULTS
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [2]:
tablas = {}
for r in api('league_standing', season='eq.2026'):
    if None in (r['played'], r['goals_for'], r['goals_against'], r['points']):
        continue
    tablas.setdefault(r['table_name'], []).append(m.TableRow(r['played'], r['goals_for'], r['goals_against'], r['points']))
for nombre, filas in tablas.items():
    print(f"{nombre:30s} {len(filas):2d} equipos · exponente pitagórico {m.calibrate_pythagorean(filas):.2f}")

Clausura - Grupo A             15 equipos · exponente pitagórico 1.39
Clausura - Grupo B             15 equipos · exponente pitagórico 1.55
Apertura - Zona A              15 equipos · exponente pitagórico 0.91
Apertura - Zona B              15 equipos · exponente pitagórico 1.18
Tabla Anual - Tabla Anual      30 equipos · exponente pitagórico 1.30


In [3]:
juegos = league_games()
QUEMA = 60  # los primeros partidos acomodan el Elo; no se evalúan
empates = sum(result(g) == 'E' for g in juegos[:QUEMA]) / QUEMA
filas = []
for k in (10, 15, 20, 25, 30, 40):
    for v in (0, 50, 100, 150):
        probs = elo_probabilities(juegos, k, v, empates)
        ll = statistics.mean(log_loss(p, result(g)) for p, g in zip(probs[QUEMA:], juegos[QUEMA:]))
        filas.append((ll, k, v))
filas.sort()
print(f"{len(juegos)} partidos, {len(juegos) - QUEMA} evaluados, tasa de empates de arranque {empates:.2f}")
for ll, k, v in filas[:5]:
    print(f"K={k:3d} V={v:3d} · log-loss {ll:.4f}")
base = elo_probabilities(juegos, 20, 100, empates)
ll_base = statistics.mean(log_loss(p, result(g)) for p, g in zip(base[QUEMA:], juegos[QUEMA:]))
print(f"Parámetros del pipeline (K=20, V=100): log-loss {ll_base:.4f}")

464 partidos, 404 evaluados, tasa de empates de arranque 0.30
K= 30 V= 50 · log-loss 1.0539
K= 25 V= 50 · log-loss 1.0542
K= 40 V= 50 · log-loss 1.0555
K= 20 V= 50 · log-loss 1.0556
K= 15 V= 50 · log-loss 1.0585
Parámetros del pipeline (K=20, V=100): log-loss 1.0718


**Lectura.** Si el mejor par mejora poco al de arranque, conviene dejar K = 20 y V = 100 (son estándar y el sitio los muestra). La diferencia entre los primeros pares de la grilla es la incertidumbre de calibrar con una sola temporada.